In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, r2_score
import warnings
from tabulate import tabulate

warnings.filterwarnings('ignore')

# --- Custom Metrics (Same as XGB) ---
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# --- Load Data ---
try:
    df = pd.read_excel("CCCdataset4.xlsx")
except FileNotFoundError:
    print("Error: CCCdataset.xlsx not found. Please ensure the file is in the correct directory.")
    raise

input_vars = ['CC', 'FA', 'CA', 'WC', 'P']
output_vars = ['pc', 'fc', 'Ec', 'e']

# --- Storage for models and scalers (Added for MOO compatibility) ---
bpnn_models = {}
bpnn_x_scalers = {}
bpnn_y_scalers = {}
performance_df = pd.DataFrame(columns=['Target', 'RMSE', 'MAPE'])
all_best_params = []

# --- Optuna Objective with K-fold (Consistent with XGB) ---
def objective(trial, X, y):
    n_layers = trial.suggest_int('n_layers', 1, 3)
    hidden_layer_sizes = tuple([trial.suggest_int(f'n_units_l{i}', 20, 200) for i in range(n_layers)])
    params = {
        'hidden_layer_sizes': hidden_layer_sizes,
        'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
        'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 1e-1, log=True),
        'alpha': trial.suggest_float('alpha', 1e-5, 1e-1, log=True),
        'max_iter': 1000,
        'random_state': 42
    }

    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_scores = []
    
    for train_idx, val_idx in kfold.split(X):
        X_train_fold, X_val_fold = X[train_idx], X[val_idx]
        y_train_fold, y_val_fold = y[train_idx], y[val_idx]

        model = MLPRegressor(**params)
        model.fit(X_train_fold, y_train_fold)

        val_pred = model.predict(X_val_fold)
        rmse = np.sqrt(mean_squared_error(y_val_fold, val_pred))
        rmse_scores.append(rmse)
    
    return np.mean(rmse_scores)

# --- Main Loop (Same structure as XGB with model storage added) ---
for target in output_vars:
    df_clean = df[input_vars + [target]].dropna(subset=[target])

    if len(df_clean) < 10:
        print(f"Skipping target {target} due to insufficient data.")
        continue

    X = df_clean[input_vars]
    y = df_clean[target].values.reshape(-1, 1)

    # --- Scale Data ---
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y).ravel()

    # --- Split Data (80/20) ---
    X_train_full, X_test, y_train_full, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

    # --- Hyperparameter Optimization ---
    study = optuna.create_study(direction='minimize')
    study.optimize(lambda trial: objective(trial, X_train_full, y_train_full), n_trials=80, show_progress_bar=False)

    # --- Get Best Params ---
    best_params = study.best_params
    # Remove n_layers from best_params since it's not an MLP parameter
    n_layers = best_params.pop('n_layers')
    # Create hidden_layer_sizes tuple
    best_params['hidden_layer_sizes'] = tuple([best_params.pop(f'n_units_l{i}') for i in range(n_layers)])
    best_params.update({
        'max_iter': 1000,
        'random_state': 42
    })
    all_best_params.append({'Target': target, 'n_layers': n_layers, **best_params})

    # --- Train Final Model on Full Training Data ---
    final_model = MLPRegressor(**best_params)
    final_model.fit(X_train_full, y_train_full)
    
    # --- Store models and scalers for MOO compatibility ---
    bpnn_models[target] = final_model
    bpnn_x_scalers[target] = scaler_X
    bpnn_y_scalers[target] = scaler_y

    # --- Predictions ---
    y_pred_test_scaled = final_model.predict(X_test)
    y_pred_train_scaled = final_model.predict(X_train_full)

    # --- Inverse Scaling ---
    y_pred_test = scaler_y.inverse_transform(y_pred_test_scaled.reshape(-1, 1)).ravel()
    y_pred_train = scaler_y.inverse_transform(y_pred_train_scaled.reshape(-1, 1)).ravel()
    y_test_original = scaler_y.inverse_transform(y_test.reshape(-1, 1)).ravel()
    y_train_full_original = scaler_y.inverse_transform(y_train_full.reshape(-1, 1)).ravel()

    # --- Calculate Metrics (Same as XGB) ---
    performance_df = pd.concat([performance_df, pd.DataFrame([{
        'Target': target,
        'RMSE': np.sqrt(mean_squared_error(y_test_original, y_pred_test)),
        'MAPE': mape(y_test_original, y_pred_test),
    }])], ignore_index=True)

  

# --- Results Display (Same format as XGB) ---
print("\n=== Performance Metrics ===")
print(performance_df)

[I 2025-09-01 20:21:17,682] A new study created in memory with name: no-name-5f438187-5d8b-4752-a201-e26ed8bb462a
[I 2025-09-01 20:21:18,212] Trial 0 finished with value: 0.2167544042693661 and parameters: {'n_layers': 3, 'n_units_l0': 168, 'n_units_l1': 136, 'n_units_l2': 53, 'activation': 'relu', 'learning_rate_init': 0.0024752363079476106, 'alpha': 0.0001234400434628023}. Best is trial 0 with value: 0.2167544042693661.
[I 2025-09-01 20:21:18,390] Trial 1 finished with value: 1.5378509049286357 and parameters: {'n_layers': 3, 'n_units_l0': 188, 'n_units_l1': 109, 'n_units_l2': 78, 'activation': 'tanh', 'learning_rate_init': 0.07564505491639491, 'alpha': 7.261190780117192e-05}. Best is trial 0 with value: 0.2167544042693661.
[I 2025-09-01 20:21:18,912] Trial 2 finished with value: 0.25040344369306655 and parameters: {'n_layers': 3, 'n_units_l0': 52, 'n_units_l1': 51, 'n_units_l2': 45, 'activation': 'tanh', 'learning_rate_init': 0.0005310926601982354, 'alpha': 0.03328005006997106}. Bes


=== Performance Metrics ===
  Target       RMSE       MAPE
0     pc  15.962281   0.507863
1     fc   2.745571   5.554466
2     Ec   1.740835   6.444290
3      e   0.000390  10.008557


In [2]:
import numpy as np                                           # Numerical Python for numerical operations                                        
import pandas as pd                                          #Python Data Analysis Library
import matplotlib.pyplot as plt 
from sklearn.preprocessing import StandardScaler             #Scikit for Machine Learning

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from pymoo.algorithms.moo.nsga3 import NSGA3                 #Non-dominated Sorting Genetic Algorithm III

from pymoo.util.ref_dirs import get_reference_directions
from pymoo.optimize import minimize                          # minimize   returns a result object res
from pymoo.core.problem import Problem                       # problem is to define deceision variables, objectives, and constraints  
from pymoo.operators.crossover.sbx import SBX               # SBX is Simulated Binary Crossover (combine parents to give child)
from pymoo.operators.mutation.pm import PM                  #polynomial mutation to purturb (modify) the solution 
from pymoo.operators.sampling.rnd import FloatRandomSampling    #Sample values uniformly at random for each decision variable within bounds
from pymoo.termination import get_termination
import warnings
warnings.filterwarnings('ignore')                         # ignore ,default, always, error etc 

In [3]:
# Cost and CO2 Emission Calculations
material_costs = {
    'CC': 0.0631,  # Cement cost ($/kg)
    'FA': 0.021,  # Fine aggregate cost ($/kg)
    'CA': 0.017,  # Coarse aggregate cost ($/kg)
    'WC': 0.000691  # Water cost ($/kg)
}

transport_cost_per_km_kg = 0.000112   # $/kg/km
transport_distance = {
    'CC': 120.9,  # Cement (km)
    'FA': 63.4,  # Fine aggregate (km)
    'CA': 34.9,  # Coarse aggregate (km)
    'WC': 0  # Water (km)
}

compaction_cost_per_minute = 1.0377  # $/minutes
material_densities = {
    'CC': 3150,
    'FA': 2650,
    'CA': 2700,
    'WC': 1000
}

co2_factors = {            #kg CO2 /kg
    'CC': 0.82,
    'FA': 0.0036,
    'CA': 0.007,
    'WC': 0.000181
}

compaction_co2_per_minute = 4.177
transport_co2_per_km_kg = 0.000078

def calculate_cost(mix, casting_pressure):
    material_cost = (mix[0] * material_costs['CC'] +
                    mix[1] * material_costs['FA'] +
                    mix[2] * material_costs['CA'] +
                    mix[3] * material_costs['WC'])
    
    transport_cost = (mix[0] * transport_distance['CC'] +  
                    mix[1] * transport_distance['FA'] + 
                    mix[2] * transport_distance['CA'] +  
                    mix[3] * transport_distance['WC']) * transport_cost_per_km_kg         

    P = mix[4]  # casting pressure
    compaction_cost = 2 * compaction_cost_per_minute if P > 0 else 0.75 * compaction_cost_per_minute

    return material_cost + transport_cost + compaction_cost
    
def calculate_co2(mix, casting_pressure):
    material_co2 = (mix[0] * co2_factors['CC'] +
                   mix[1] * co2_factors['FA'] +
                   mix[2] * co2_factors['CA'] +
                   mix[3] * co2_factors['WC'])

    transport_co2 = (mix[0] * transport_distance['CC'] +  
                    mix[1] * transport_distance['FA'] + 
                    mix[2] * transport_distance['CA'] +  
                    mix[3] * transport_distance['WC']) * transport_co2_per_km_kg         

    P = mix[4]  # casting pressure
    compaction_co2 = 2 * compaction_co2_per_minute if P > 0 else 0.75 * compaction_co2_per_minute

    return material_co2 + transport_co2 + compaction_co2

In [14]:
# Multi-Objective Optimization Problem Definition with Tight Strength Constraint topsis with weights 
class ConcreteMixProblem(Problem):
    def __init__(self, bpnn_models, x_scalers, y_scalers, target_strength=None):      #, casting_pressure=None, casting_time=None
        
        n_var = 5  # CC, FA, CA, WC, P
        xl = [190, 480, 806, 100, 0]  # Lower bounds (kg/m3)
        xu = [612, 945, 1175, 266, 15]  # Upper bounds (kg/m3)
        
        # 3 objectives: minimize cost, minimize CO2, maximize density
        n_obj = 3
        
        # Constraints:
        # 1. Volume constraint (sum of volumes = 1 m3)
        # 2. Water-cement ratio (0.3 <= w/c <= 0.6)
        # 3. Fine aggregate ratio (0.3 <= FA/(FA+CA) <= 0.5)
        # 4. Strength lower bound (fc >= target_strength)
        # 5. Strength upper bound (fc <= target_strength + 1)
        n_constr = 5
        
        super().__init__(n_var=n_var, n_obj=n_obj, n_constr=n_constr, xl=xl, xu=xu)
        
        self.bpnn_models = bpnn_models
        self.x_scalers = x_scalers
        self.y_scalers = y_scalers
        self.target_strength = target_strength
        #self.casting_pressure = casting_pressure
       # self.casting_time = casting_time
        
    def _evaluate(self, X, out, *args, **kwargs):
        n_samples = X.shape[0]
        f = np.zeros((n_samples, self.n_obj))
        g = np.zeros((n_samples, self.n_constr))
        
        for i in range(n_samples):
            mix = X[i]
            cc, fa, ca, wc, P = mix
            
            full_input = np.array([cc, fa, ca, wc, P])     #, self.casting_time, self.casting_pressure
            
            preds = {}
            for target, model in self.bpnn_models.items():
                scaled_input = self.x_scalers[target].transform(full_input.reshape(1, -1))
                scaled_pred = model.predict(scaled_input)
                preds[target] = self.y_scalers[target].inverse_transform(scaled_pred.reshape(-1, 1))[0, 0]
            
            # Objectives
            f[i, 0] = calculate_cost(full_input, P)  # Cost , self.casting_time, self.casting_pressure
            f[i, 1] = calculate_co2(full_input, P)  # CO2 , self.casting_time,  self.casting_pressure
            f[i, 2] = -preds['pc']  # Negative density (to maximize)
            
            # Constraints
            volumes = np.array([
                cc / material_densities['CC'],
                fa / material_densities['FA'],
                ca / material_densities['CA'],
                wc / material_densities['WC']
            ])
            total_volume = np.sum(volumes)
            g[i, 0] = abs(total_volume - 1) - 0.01  # Volume constraint
            
            wc_ratio = wc / cc
            g[i, 1] = max(0.35 - wc_ratio, wc_ratio - 0.65, 0)  # W/C ratio
            
            fa_ratio = fa / (fa + ca)
            g[i, 2] = max(0.3 - fa_ratio, fa_ratio - 0.45, 0)  # FA ratio
            
            # Strength constraints (target ≤ fc ≤ target + 3)
            g[i, 3] = self.target_strength - preds['fc']  # Lower bound (fc >= target)
            g[i, 4] = preds['fc'] - (self.target_strength + 2.5)  # Upper bound (fc <= target + 1)
            
        out["F"] = f
        out["G"] = g

def optimize_mix(target_strength, n_gen=100, pop_size=450):     #casting_time, casting_pressure, 
    # Create the problem instance
    problem = ConcreteMixProblem(
        bpnn_models=bpnn_models,
        x_scalers=bpnn_x_scalers,
        y_scalers=bpnn_y_scalers,
        target_strength=target_strength,
        #casting_pressure=casting_pressure,
        #casting_time=casting_time
    )
    
    # Set up the reference directions for NSGA3
    ref_dirs = get_reference_directions("das-dennis", 3, n_partitions=25)    #3 is no of objectives
    
    # Configure the algorithm
    algorithm = NSGA3(
        pop_size=pop_size,
        ref_dirs=ref_dirs,
        sampling=FloatRandomSampling(),
        crossover=SBX(prob=0.9, eta=10),
        mutation=PM(eta=20),
        eliminate_duplicates=True
    )
    
    # Set termination condition
    termination = get_termination("n_gen", n_gen)
    
    # Run the optimization
    res = minimize(
        problem,
        algorithm,
        termination,
        seed=42,
        verbose=False
    )
    
    return res, problem

def get_predictions(x, problem):
    """Predict properties for a concrete mix using trained BPNN models"""
    full_input = x
    preds = {}
    for target, model in problem.bpnn_models.items():
        scaled_input = problem.x_scalers[target].transform(full_input.reshape(1, -1))
        scaled_pred = model.predict(scaled_input)
        preds[target] = problem.y_scalers[target].inverse_transform(scaled_pred.reshape(-1, 1))[0, 0]
    return preds

def select_optimal_solution(res, problem, weights=None):
  
    F = res.F
    G = res.G
    X = res.X
    
    # Default weights (equal weighting)
    if weights is None:
        weights = {'cost': 1/3, 'co2': 1/3, 'density': 1/3}
    
    # Convert weights to array in correct order (cost, co2, density)
    weight_array = np.array([weights.get('cost', 0), 
                            weights.get('co2', 0), 
                            weights.get('density', 0)])
    # Normalize weights to sum to 1
    weight_array = weight_array / np.sum(weight_array)
    
    # Get feasible solutions
    feasible_mask = np.all(G <= 0, axis=1)
    feasible_F = F[feasible_mask]
    feasible_X = X[feasible_mask]
    
    if len(feasible_F) == 0:
        print("Warning: No feasible solutions found. Returning closest to feasible.")
        cv = np.sum(np.maximum(G, 0), axis=1)
        best_idx = np.argmin(cv)
        return [X[best_idx]], [F[best_idx]], [0.0], [get_predictions(X[best_idx], problem)]
    
    # Normalize objectives to [0, 1] range
    norm_F = (feasible_F - feasible_F.min(axis=0)) / (feasible_F.max(axis=0) - feasible_F.min(axis=0) + 1e-10)
    
    # Apply weights to normalized objectives
    weighted_norm_F = norm_F * weight_array
    
    # Determine ideal and negative-ideal solutions
    ideal = np.array([
        np.min(weighted_norm_F[:, 0]),  # Cost (minimize)
        np.min(weighted_norm_F[:, 1]),  # CO2 (minimize)
        np.max(weighted_norm_F[:, 2])   # Density (maximize)
    ])
    
    neg_ideal = np.array([
        np.max(weighted_norm_F[:, 0]),  # Cost
        np.max(weighted_norm_F[:, 1]),  # CO2
        np.min(weighted_norm_F[:, 2])    # Density
    ])
    
    # Calculate distances with weights
    d_pos = np.sqrt(np.sum((weighted_norm_F - ideal)**2, axis=1))
    d_neg = np.sqrt(np.sum((weighted_norm_F - neg_ideal)**2, axis=1))
    
    # Calculate TOPSIS score
    topsis_score = d_neg / (d_pos + d_neg + 1e-10)
    
    # Get top 10 solutions
    top10_idx = np.argsort(topsis_score)[-5:][::-1]
    
    top10_X = feasible_X[top10_idx]
    top10_F = feasible_F[top10_idx]
    top10_scores = topsis_score[top10_idx]
    top10_preds = [get_predictions(x, problem) for x in top10_X]
    
    return top10_X, top10_F, top10_scores, top10_preds

def design_concrete_mix(target_strength, weights=None):
    print(f"\nDesigning concrete mix for target strength: {target_strength} MPa")
    
    res, problem = optimize_mix(target_strength, n_gen=100, pop_size=450)
    top_X, top_F, top_scores, top_preds = select_optimal_solution(res, problem, weights=weights)
    
    results = []
    for i, (mix, obj, score, preds) in enumerate(zip(top_X, top_F, top_scores, top_preds)):
        cost = obj[0]
        co2 = obj[1]
        density = -obj[2]
        
        strength_met = (preds['fc'] >= target_strength) and (preds['fc'] <= target_strength + 2.5)
        strength_status = "✔" if strength_met else "✖"
        
        results.append({
            'Rank': i+1,
            'TOPSIS_Score': f"{score:.4f}",
            'Mix Proportions': {
                'Cement': f"{mix[0]:.1f} kg/m³",
                'Fine Aggregate': f"{mix[1]:.1f} kg/m³",
                'Coarse Aggregate': f"{mix[2]:.1f} kg/m³",
                'WC': f"{mix[3]:.3f}",
                'Pressure': f"{mix[4]:.1f} MPa"
            },
            'Objectives': {
                'Cost': f"${cost:.2f}",
                'CO2': f"{co2:.2f} kg",
                'Density': f"{density:.1f} kg/m³"
            },
            'Properties': {
                'Strength': f"{preds['fc']:.2f} MPa {strength_status}",
                'w/c Ratio': f"{(mix[3]/(mix[0])):.3f}"
            },
            'Constraints': {
                'Volume': f"{sum([mix[i]/material_densities[m] for i, m in enumerate(['CC', 'FA', 'CA', 'WC'])]):.3f} m³",
                'FA Ratio': f"{(mix[1]/(mix[1]+mix[2])):.3f}"
            }
        })
    
    print_results_table(results)
    return results

def print_results_table(results):
    """Prints the results in a formatted table"""
    from tabulate import tabulate
    
    table_data = []
    headers = [
        "Rank", "Cement (kg/m³)", "FA (kg/m³)", "CA (kg/m³)", 
        "WC", "Pressure (MPa)", "Cost ($)", "CO2 (kg)", 
        "Density (kg/m³)", "TOPSIS Score", "Strength (MPa)", 
        "Volume (m³)", "FA Ratio"
    ]
    
    for res in results:
        row = [
            res['Rank'],
            res['Mix Proportions']['Cement'].split()[0],
            res['Mix Proportions']['Fine Aggregate'].split()[0],
            res['Mix Proportions']['Coarse Aggregate'].split()[0],
            res['Mix Proportions']['WC'],
            res['Mix Proportions']['Pressure'].split()[0],
            res['Objectives']['Cost'].split("$")[1],
            res['Objectives']['CO2'].split()[0],
            res['Objectives']['Density'].split()[0],
            res['TOPSIS_Score'],
            res['Properties']['Strength'].split()[0],
            res['Constraints']['Volume'],
            res['Constraints']['FA Ratio']
        ]
        table_data.append(row)
    
    print(tabulate(table_data, headers=headers, tablefmt="plain", floatfmt=".3f"))

# Example usage
# Example usage with custom weights
if __name__ == "__main__":
    # Define your weights (must sum to 1 or they will be normalized)
    custom_weights = {
        'cost': 0.33,    # Higher importance to cost
        'co2': 0.33,     # Medium importance to CO2
        'density': 0.33  # Lower importance to density
    }
    
    results = design_concrete_mix(target_strength=43, weights=custom_weights)
    

    


Designing concrete mix for target strength: 43 MPa
  Rank    Cement (kg/m³)    FA (kg/m³)    CA (kg/m³)       WC    Pressure (MPa)    Cost ($)    CO2 (kg)    Density (kg/m³)    TOPSIS Score    Strength (MPa)  Volume (m³)      FA Ratio
     1           312.800       770.700      1142.700  177.732            15.000      71.720     285.570           2421.500           0.852            43.060  0.991 m³            0.403
     2           313.000       770.700      1142.700  177.732            15.000      71.740     285.700           2421.600           0.851            43.090  0.991 m³            0.403
     3           311.300       773.100      1172.200  165.757            15.000      72.280     284.600           2418.900           0.797            43.410  0.990 m³            0.397
     4           318.000       784.300      1118.100  179.569            15.000      71.990     289.720           2425.200           0.784            43.310  0.991 m³            0.412
     5           321.700    

In [10]:
# === Example Run ===

# Define an example concrete mix: [CC, FA, CA, WC, P]
# Units are in kg for materials, MPa for pressure (P)
example_mix = [445.1, 479, 1023.3, 289.1, 5.9]  # realistic compression-cast concrete mix

# Prepare scaled input for prediction (reshape to 2D)
X_example = np.array(example_mix).reshape(1, -1)

# Predict each target variable
predictions = {}
for target in output_vars:
    model = bpnn_models.get(target)
    x_scaler = bpnn_x_scalers.get(target)
    y_scaler = bpnn_y_scalers.get(target)
    
    if model is None or x_scaler is None or y_scaler is None:
        print(f"Skipping prediction for {target} (model or scaler missing).")
        continue

    X_scaled = x_scaler.transform(X_example)
    y_pred_scaled = model.predict(X_scaled)
    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()[0]
    predictions[target] = y_pred

# Display predicted properties
print("\n--- Predicted Concrete Properties ---")
print(f"(pc): {predictions.get('pc', 'N/A'):.2f} MPa")
print(f"(fc): {predictions.get('fc', 'N/A'):.2f} MPa")
print(f"(Ec): {predictions.get('Ec', 'N/A'):.2f} GPa")
print(f"(e): {predictions.get('e', 'N/A'):.4f}")

# Calculate cost and CO₂ emissions
cost = calculate_cost(example_mix, example_mix[4])
co2 = calculate_co2(example_mix, example_mix[4])

print(f"\n--- Cost and CO₂ Emissions ---")
print(f"Total Cost: ${cost:.2f}")
print(f"Total CO₂ Emissions: {co2:.2f} kg CO₂")


--- Predicted Concrete Properties ---
(pc): 2267.65 MPa
(fc): 34.41 MPa
(Ec): 23.73 GPa
(e): 0.0046

--- Cost and CO₂ Emissions ---
Total Cost: $71.24
Total CO₂ Emissions: 391.63 kg CO₂


In [15]:
# --- Save models and data to pickle file ---
import pickle
import os

# Create the data dictionary with the exact keys expected by your loading code
save_data = {
    'bpnn_models': bpnn_models,
    'bpnn_x_scalers': bpnn_x_scalers,
    'bpnn_y_scalers': bpnn_y_scalers,
    'input_vars': input_vars,
    'output_vars': output_vars,
    'material_costs': material_costs,
    'transport_distance': transport_distance,
    'material_densities': material_densities,
    'co2_factors': co2_factors
}

# Save to pickle file
with open('concrete_models.pkl', 'wb') as f:
    pickle.dump(save_data, f)

print(f"Models and data saved to concrete_models.pkl")
print(f"File size: {os.path.getsize('concrete_models.pkl') / 1024:.2f} KB")
print(f"Keys saved: {list(save_data.keys())}")
print(f"Models available for targets: {list(bpnn_models.keys())}")

# --- Verify the saved file can be loaded correctly ---
print("\n=== Verifying saved file ===")
try:
    with open('concrete_models.pkl', 'rb') as f:
        loaded_data = pickle.load(f)
    
    # Check if all expected keys are present
    expected_keys = ['bpnn_models', 'bpnn_x_scalers', 'bpnn_y_scalers', 'input_vars', 
                    'output_vars', 'material_costs', 'transport_distance', 
                    'material_densities', 'co2_factors']
    
    missing_keys = [key for key in expected_keys if key not in loaded_data]
    if missing_keys:
        print(f"Warning: Missing keys in saved file: {missing_keys}")
    else:
        print("All expected keys are present in the saved file")
        
    # Test that models can be used for prediction
    test_sample = pd.DataFrame([[350, 150, 1000, 180, 0.4]], columns=input_vars)
    target_to_test = list(loaded_data['bpnn_models'].keys())[0]
    
    # Scale input
    X_scaled = loaded_data['bpnn_x_scalers'][target_to_test].transform(test_sample)
    # Make prediction
    y_pred_scaled = loaded_data['bpnn_models'][target_to_test].predict(X_scaled)
    # Inverse scale
    y_pred = loaded_data['bpnn_y_scalers'][target_to_test].inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).ravel()
    
    print(f"Test prediction for {target_to_test}: {y_pred[0]:.4f}")
    print("✓ File verification successful!")
    
except Exception as e:
    print(f"Error during verification: {e}")

Models and data saved to concrete_models.pkl
File size: 2447.98 KB
Keys saved: ['bpnn_models', 'bpnn_x_scalers', 'bpnn_y_scalers', 'input_vars', 'output_vars', 'material_costs', 'transport_distance', 'material_densities', 'co2_factors']
Models available for targets: ['pc', 'fc', 'Ec', 'e']

=== Verifying saved file ===
All expected keys are present in the saved file
Test prediction for pc: 2033.1328
✓ File verification successful!
